# Дослідницька програма для базлайну 91.13% (run_20260630_235356)

Ноутбук відповідає на запити керівника: **(1)** наочний post-mortem помилок 2→7;
**(2)** шість досліджень — інші датасети MNIST, залежність концепту від порядку/кількості/аугментації
навчальних прикладів, кроки редукції до сталого концепту, аналіз параметрів концептів.

**Як користуватися**
- Kernel: **natural-agi** (venv проєкту). Якщо його немає в списку — виконайте один раз у терміналі:
  `natural-agi/bin/python -m ipykernel install --user --name natural-agi --display-name "natural-agi"`
- Вся логіка — у пакеті `src/training/supervisor_experiments/`; клітинки лише викликають функції та показують результат.
- 🔴 — руйнівні клітинки (зупиняють/чистять Neo4j). Вони виконуються **тільки** якщо ви явно поставите `CONFIRM = "yes"`, і перевіряють наявність бекапу.
- ⚠️ Пастка Jupyter: відредагована клітинка **не** виконується сама — після зміни коду клітинку треба запустити повторно.
- Довгі клітинки друкують час виконання і зберігають артефакти на диск (повторний запуск безпечний).

**Секції:** 0 — налаштування · 1 — post-mortem 2→7 · 2 — параметри концептів (S6) · 3 — збіжність редукції (S5) ·
4 — порядок пред'явлення (S2) · 5 — кількість прикладів (S3) · 6 — датасети MNIST (S1) · 7 — 🔴 ретрени (S3/S4) · 8 — зведення.

In [ ]:
import logging, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _c in (_here, *_here.parents):
    if (_c / "src" / "training" / "supervisor_experiments").exists():
        REPO = _c
        sys.path.insert(0, str(_c / "src" / "training"))
        break
else:
    raise RuntimeError("Запустіть Jupyter з кореня репозиторію або з src/training/")

logging.disable(logging.INFO)  # приглушити DEBUG/INFO продукційних модулів
%matplotlib inline

from supervisor_experiments import infra, report
from supervisor_experiments import postmortem as pm

RUN_ID = pm.RUN_ID
print(f"REPO = {REPO}")
print(f"Базлайн: {RUN_ID} (91.13%)")

### 0.1 Перевірка середовища

Очікування: Neo4j запущений і містить **13 концептів** та **8 685 графів зображень** базлайн-прогону
(`baseline_state_intact: True`). Для Секцій 1–5 достатньо самого Neo4j; Kafka/Nuclio потрібні лише
для Секцій 6–7 (`make start_services` + `docker start $(docker ps -aq --filter name=nuclio-nuclio)`).

In [ ]:
health = infra.health_check()
for k, v in health.items():
    print(f"  {k}: {v}")
assert health["neo4j_container_up"], "Neo4j не запущений: docker start naturalagi-neo4j-1"
assert health["baseline_state_intact"], "Стан Neo4j НЕ відповідає базлайну — див. Секцію 7 (відновлення з бекапа)"
print("\n✅ Середовище готове (Tier A)")

### 0.2 🔴 Бекап Neo4j (обов'язковий шлюз перед Секцією 7)

Холодний бекап volume: Neo4j зупиняється на ~1 хв, у `backups/neo4j_baseline_2026-07-17.tar.gz`
зберігається весь стан (концепти + 8 685 графів зображень). Без цього файлу руйнівні клітинки Секції 7 відмовляться працювати.

In [ ]:
CONFIRM = "no"  # ← поставте "yes", щоб зробити бекап (Neo4j зупиниться на ~1 хв)

if CONFIRM == "yes":
    infra.backup_neo4j(CONFIRM)
else:
    print(f"Бекап існує: {infra.BACKUP_PATH.exists()} ({infra.BACKUP_PATH})")
    print('Щоб створити/оновити: CONFIRM = "yes" і перезапустіть клітинку.')

## Секція 1 — Post-mortem: чому «2» падають у «7»

**Механізм (підтверджено на всіх 111 промахах 2→7):** двоступеневий, зі спільною першопричиною —
втратою якірних точок при побудові графа (ще до редукції):

1. Пре-фільтр складності (`classification_orchestrator.py:97-102`) відкидає концепти зі
   `складність > складність_зображення` **до** порівняння. Збіднений скелет «2» має медіанну складність 21,
   а «повний» концепт 2_2 — 24 → у **77.5%** випадків 2_2 взагалі не бере участі у WTA-конкуренції.
2. Спрощений 2_1 (складність 13) конкурує, але програє 7_1 за сирою схожістю (середній розрив 0.14):
   вцілілі точки стоять «не там» (OUT-of-range за просторовими ознаками).

Кожна клітинка нижче відтворює один крок аналізу продукційним кодом. Підсумковий документ:
[`researches/two_to_seven_postmortem.md`](../../researches/two_to_seven_postmortem.md).

In [ ]:
params = pm.load_params()
concepts = pm.load_run_concepts()
confusion = pm.load_confusion(expected="2", predicted="7")
print(f"Концептів у снепшоті прогону: {len(concepts)}")
print(f"Промахів 2→7 у {RUN_ID}: {len(confusion)}")
confusion.head(3)

**1.1 Як виглядають ці «2»** (перші 12 із 111):

In [ ]:
fig = pm.show_images(confusion, n=12)
report.save_figure(fig, "two_to_seven/misclassified_2_examples.png")

**1.2 Продукційний ранкінг еталонного прикладу** — `mnist_test_2_00766` (канонічна «2» із петлею).
Читання виводу: `raw_sim` — сира GED-схожість; `adjusted = raw + 0.10·log₂(складність)`;
концепт **відсутній у списку** ⇔ його зрізав пре-фільтр складності (тут: 2_2 і 8_1).
Нижче — per-feature розбір програшу 2_1 (OUT-of-range ⇒ повний NO_MATCH за Family E).

In [ ]:
print(pm.explain_cli(pm.PRIMARY_EXEMPLAR, expected="2"))

**1.3 Бакет-аналіз усіх 111 промахів.** Бакети: **A** — жоден 2-концепт не пройшов пре-фільтр;
**B** — конкурував, програв за сирою схожістю; **C** — виграв raw, програв λ-ранжування.
Додатково рахуємо, чи був відфільтрований саме 2_2 (`c22_prefiltered`).

In [ ]:
buckets = pm.bucket_all(list(confusion.image_id), concepts, params, expected="2")
buckets = buckets.merge(confusion[["image_id", "image_path"]], on="image_id")
buckets.to_csv(report.FIGURES / "two_to_seven" / "bucket_analysis.csv", index=False)
display(pm.bucket_summary(buckets))
n22 = int(buckets.c22_prefiltered.sum())
print(f"2_2 відфільтровано пре-фільтром: {n22}/{len(buckets)} ({n22/len(buckets)*100:.1f}%)")
print(f"Збіг прогнозу інструмента з прогоном: "
      f"{int((buckets.predicted_tool == '7_1').sum())}/{len(buckets)}")
fig = pm.plot_buckets(buckets)
report.save_figure(fig, "two_to_seven/bucket_bar.png")

**1.4 Триптих для керівника:** зображення → граф до редукції → концепти 2_2 (відфільтровано) і 7_1 (переможець).

In [ ]:
row = confusion[confusion.image_id == pm.PRIMARY_EXEMPLAR].iloc[0]
fig = pm.triptych(pm.PRIMARY_EXEMPLAR, row, concepts, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00766_triptych.png")

**1.5 Стадії побудови графа** — де саме зникають якірні точки: в оригіналі голова «2» — замкнена петля;
після бінаризації + 1px-thinning вона стає відкритим гачком (0 циклів), GNG+RDP сплющують дугу в 2 сегменти.

In [ ]:
fig = pm.construction_figure(row, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00766_construction.png")

**1.6 Контрастний приклад** — `mnist_test_2_00984`: пласка курсивна «2», «чесно» схожа на 7 (не всі промахи — жертви скелетонізації).

In [ ]:
row_c = confusion[confusion.image_id == pm.CONTRAST_EXEMPLAR].iloc[0]
fig = pm.triptych(pm.CONTRAST_EXEMPLAR, row_c, concepts, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00984_triptych.png")
fig = pm.construction_figure(row_c, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00984_construction.png")

### Висновок (Секція 1)

- Складніший концепт «2» не програє WTA-конкуренцію — **він у неї не потрапляє**: пре-фільтр складності
  зрізає 2_2 у 86/111 випадків (77.5%), бо збіднений скелет має складність ≤ 21 < 24.
- Причина збіднення видима на фігурі стадій: **петля голови «2» руйнується на бінаризації/thinning**
  (0 циклів у скелеті), GNG+RDP довершують спрощення. Це і є «втрата якірних точок до редукції».
- Навіть уцілілий 2_1 програє 7_1 за сирою схожістю (розрив 0.14) — вцілілі точки геометрично зміщені
  (OUT-of-range за `distance_to_centroid`, `normalized_x`).
- λ-ранжування невинне: бакет C = 0. Головний важіль — **скелетонізація, що зберігає петлі**;
  пом'якшення пре-фільтра — лише симптоматичне (властивісне, без хардкоду concept_id).

Повний текст: `researches/two_to_seven_postmortem.md`.

## Секція 2 — Параметри концептів і компресія (S6)

⏳ *Буде додано наступною фазою збірки.*

## Секція 3 — Кроки редукції до сталого концепту (S5)

⏳ *Буде додано наступною фазою збірки.*

## Секція 4 — Залежність від порядку пред'явлення (S2)

⏳ *Буде додано наступною фазою збірки.*

## Секція 5 — Залежність від кількості прикладів (S3, офлайн)

⏳ *Буде додано наступною фазою збірки.*

## Секція 6 — Інші датасети MNIST (S1)

⏳ *Буде додано наступною фазою збірки.*

## Секція 7 — 🔴 Ретрени: кількість прикладів (downstream) і аугментація (S3/S4)

⏳ *Буде додано наступною фазою збірки.*

## Секція 8 — Зведений звіт

⏳ *Буде додано наступною фазою збірки.*